# Implementing `Co-Clustering Triples from Open Information Extraction` From Scratch

### Setting Up Library

In [3]:
import torch
import random
import warnings
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm
from itertools import combinations
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.cluster import AgglomerativeClustering

### 1. Data Simulation and Embedding Generation

In [4]:
def simulate_numerical_data():
    base_facts = [
        {'id': 1, 's': 'India', 'p': 'has a population of', 'o': 1400000000},
        {'id': 2, 's': 'Mount Everest', 'p': 'has a height of', 'o': 8848},
    ] # Simulated numerical data
    
    generated_facts = list(base_facts)
    aliases = {
        'India': ['Bharat', 'Hindustan'],
        'Mount Everest': ['Sagarmatha', 'Everest'],
    }
    pred_phrases = {'has a population of': 'population stands at',
                    'has a height of': 'is tall'}# predicted phrases
    for fact in base_facts:
        new_fact = fact.copy()# copying the fact
        new_fact['s'] = aliases.get(fact['s'], [fact['s']]) # randomly selecting an alias
        new_fact['p'] = pred_phrases.get(fact['p'], fact['p']) # using predicted phrase
        variation = fact['o'] * random.uniform(-0.1, 0.1) # adding variation to the object for numerical data
        new_fact['o'] = int(fact['o'] + variation) # updating the object with variation
        generated_facts.append(new_fact) # appending the new fact to the list
    generated_facts.extend([
        {'id': 3, 's': 'Brazil', 'p': 'has a GDP of', 'o': 1600000000000},
        {'id': 4, 's': 'K2', 'p': 'is located in', 'o': 8611},
    ])# adding more facts
    print("Simulated numerical dataset.")
    return generated_facts
        

- #### Building Embedding Generation

In [5]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')# loading the tokenizer
bert_model = BertModel.from_pretrained('bert-base-uncased')# loading the BERT model
BERT_DIM = bert_model.config.hidden_size # getting the BERT dimension

In [6]:
# function to get BERT embeddings for a list of texts
def get_bert_embedding(text_list):
    inputs = tokenizer(text_list, return_tensors='pt', padding=True, truncation=True, max_length=32)# tokenizing the input texts
    with torch.no_grad():
        outputs = bert_model(**inputs) # getting the BERT model outputs
    return outputs.last_hidden_state[:, 0, :].numpy() # returning the embeddings of the [CLS] token

- #### Implementing DICE Embeddings

In [7]:
class DICEEmbedder:
    def __init__(self, dimensions=100, max_val = 1e15):
        self.dimensions = dimensions# setting the dimensions for the embeddings
        self.max_val = max_val# setting the maximum value for the embeddings
    
    def get_dice_embedding(self, number):
        if not isinstance(number, (int, float)):
            return torch.zeros(1, self.dimensions) # returning zero vector for non-numerical inputs
        number = float(number) # converting the number to float
        embedding = torch.zeros(self.dimensions) # initializing the embedding vector
        scaled_num = torch.log(torch.tensor(abs(number)+ 1.0))# scaling the number
        for i in range(self.dimensions// 2):
            div_term = torch.exp(torch.tensor(i * -np.log(torch.tensor(self.max_val)) / (self.dimensions / 2)))# calculating the division term
            embedding[2*i] = torch.sin(scaled_num * div_term) # sine component
            embedding[2*i + 1] = torch.cos(scaled_num * div_term) # cosine component
        if number < 0:
            embedding[-1] = -1.0 # setting the last component to -1 for negative numbers
        return embedding.unsqueeze(0) # returning the embedding as a 2D tensor

In [8]:
# Initializing DICE embedder
DICE_DIM = 100 # setting the DICE dimensions
dice_embedder = DICEEmbedder(dimensions=DICE_DIM) # creating an instance of DICEEmbedder
print("DICE And BERT embeddings initialized with dimensions:", DICE_DIM, "and", BERT_DIM)

DICE And BERT embeddings initialized with dimensions: 100 and 768


### 2. Building PyTorch Model Architecture

In [ ]:
# building primary model from paper for numerical data
class Model3_NM(nn.Module):
    
    def __init__(self):
        super(Model3_NM, self).__init__()# initializing the model
        self.relu = nn.ReLU() # ReLU activation function
        
        # building compoonent projection layers
        self.dense_proj_bert = nn.Linear(BERT_DIM, 128) # BERT projection layer
        self.mlp_dice = nn.Linear(DICE_DIM, 32) # DICE projection layer
        
        # Path 1st: Component wise similarity
        self.mlp_e = nn.Linear(256, 64)  # 128 * 2 i.e. BERT + DICE
        self.mlp_p = nn.Linear(256, 64)  # 128 * 2
        self.mlp_o = nn.Linear(320, 64)  # (128+32) * 2
        self.fusion_original = nn.Linear(192, 64)  # 64 * 3
        self.output_original = nn.Linear(64, 1)# final output layer for original embeddings
        
        #building path 2nd: fact-level similarity
        self.mlp_f = nn.Linear(288, 128) # 128(s) + 128(p) + 32(o_dice)
        self.fusion_final = nn.Linear(256, 64) # 128 * 2
        self.output_final = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()# final activation function
    
    